A function that takes merged monthly netcdf file and converts it to seasonal

Monthly netcdf file:
- Lat
- Long
- predicted precip
- actual precip
- date
- lead time

Seasonal netcdf file:
- lat
- long
- predicted precip
- actual precip
- date
- lead time category
- (month - lead time) for seasonality
- season

In [1]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd

In [2]:
# import google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# load an example merged netcdf file
merged_monthly_file = xr.open_dataset('/content/drive/MyDrive/capstone_data/netCDF/eastern_east_africa_CanESM5_merged.nc')
merged_monthly_file_df = merged_monthly_file.to_dataframe().reset_index().dropna()

In [4]:
# seasons dictionary for month minus lead time

# if month_minus_lead is [9.5, 8.5, 7.5], then those squares are all OND short lead (0-2 months)
# the rest of the seasons must be defined accordingly by manual input
# short:0-1, med: 2-3, long: 4-6 months
# procedure: take the full monthly data for a given region, keep only the months
# of the seasons for that region. make a second season column if needed
# example: southern africa seasons = DJF, FMA
# subset the data so that there are only DJF and FMA months,
# in the season 1 column assign DJF short/med/long,
# copy the dataframe, and in the season 2 column assign FMA short/med/long,
# NA if otherwise, merge those 2 dataframes to get the final dataframe
# convert to netcdf
# if there are 3 or more seasons, adjust accordingly
'''
From our grid heatmaps, we wanted to group individual months into seasons.
For example, the OND season in eastern east africa consists of the months of October, November, and December.
We also wanted to group lead times into categories.
For example, the OND short lead category consists of lead times of 0-1 months.
We did this by looking at the month minus lead time values, visualized below.

Matrix of month minus lead time values

[[11.5 10.5  9.5  8.5  7.5  6.5  5.5  4.5  3.5  2.5  1.5  0.5]
 [10.5  9.5  8.5  7.5  6.5  5.5  4.5  3.5  2.5  1.5  0.5 -0.5]
 [ 9.5  8.5  7.5  6.5  5.5  4.5  3.5  2.5  1.5  0.5 -0.5 -1.5]
 [ 8.5  7.5  6.5  5.5  4.5  3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5]
 [ 7.5  6.5  5.5  4.5  3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5]
 [ 6.5  5.5  4.5  3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5]
 [ 5.5  4.5  3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5]
 [ 4.5  3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5 -6.5]
 [ 3.5  2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5 -6.5 -7.5]
 [ 2.5  1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5 -6.5 -7.5 -8.5]
 [ 1.5  0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5 -6.5 -7.5 -8.5 -9.5]
 [ 0.5 -0.5 -1.5 -2.5 -3.5 -4.5 -5.5 -6.5 -7.5 -8.5 -9.5 -10.5]]

Where you would imagine an x axis of lead times (0.5, 1.5, 11.5, etc.) and a y axis of months (1-12).

We would take a heatmap, then take a given square, and subtract the month
from the lead time. For example, if the month is October (10) and the lead time is 0.5 months,
then the month minus lead time is 9.5. The duplicated values are the predictions made
at the same time at the start of the season. In this example, all the values that are 9.5 and 8.5
where months are between October, November, and December are OND short lead.

Using this information, we want to take our merged monthly netcdf file for a
given model and region, and add a new column that assigns the seasonality lead time category
(i.e OND_short) for each prediction made by the given model for that region.

(Add more explanation, and fix above to be more clear and detailed)

'''
regions_seasons_dict = {
    'eastern_east_africa': {
        'OND': {'OND_short': [9.5, 8.5],
                'OND_medium': [7.5, 6.5],
                'OND_long': [5.5, 4.5, 3.5],
                'months': [10, 11, 12]},
        'MAM': {'MAM_short': [2.5, 1.5],
                'MAM_medium': [0.5, -0.5],
                'MAM_long': [-1.5, -2.5, -3.5],
                'months': [3, 4, 5]}
    },
    'lake_victoria': {
        'DJF': {'DJF_short': [11.5, 10.5, -0.5, -1.5],
                'DJF_medium': [9.5, 8.5, -2.5, -3.5],
                'DJF_long': [7.5, 6.5, 5.5, -4.5, -5.5, -6.5],
                'months': [12, 1, 2]},
         'MAM': {'MAM_short': [2.5, 1.5],
                'MAM_medium': [0.5, -0.5],
                'MAM_long': [-1.5, -2.5, -3.5],
                'months': [3, 4, 5]},
        'SON': {'SON_short': [8.5, 7.5],
                'SON_medium': [6.5, 5.5],
                'SON_long': [4.5, 3.5, 2.5],
                'months': [9, 10, 11]}
    },
    'west_africa': {
        'JAS': {'JAS_short': [],
                'JAS_medium': [],
                'JAS_long': [],
                'months': [7, 8, 9]}
    },
    'southern_africa': {
        'DJF': {'DJF_short': [],
                'DJF_medium': [],
                'DJF_long': [],
                'months': [12, 1, 2]},
        'FMA': {'FMA_short': [],
                'FMA_medium': [],
                'FMA_long': []}
    },
    'south_sudan': {
        'MJJ': {'MJJ_short': [],
                'MJJ_medium': [],
                'MJJ_long': []},
        'JAS': {'JAS_short': [],
                'JAS_medium': [],
                'JAS_long': []},
        'ASO': {'ASO_short': [],
                'ASO_medium': [],
                'ASO_long': []}
    },
    'eastern_ukraine': {
        'DJF': {'DJF_short': [],
                'DJF_medium': [],
                'DJF_long': []},
        'AMJ': {'AMJ_short': [],
                'AMJ_medium': [],
                'AMJ_long': []},
        'JA': {'JA_short': [],
               'JA_medium': [],
               'JA_long': []}
    },
    'sri_lanka': {
        'OND': {'OND_short': [],
                'OND_medium': [],
                'OND_long': []}
    }
}



In [5]:
def convert_monthly_to_seasonal(file_path, regions_seasons_dict, save_path):
  """
  This function takes a merged monthly netcdf file and converts it to seasonal

  Arguments:
  - file_path: path to merged monthly netcdf file
  - regions_seasons_dict: dictionary of regions and their seasons
  - save_path: path to save the seasonal netcdf file

  Usage Notes:
  Ensure that the file_path follows this format:
  '/content/drive/MyDrive/data/netCDF/eastern_east_africa_CanESM5_merged.nc'
  It does not matter what the netcdf file name is, as long as it follows the format of
  some_region_here_model_merged.nc

  Ensure that the save_path follows this format:
  '/content/drive/MyDrive/data/netCDF'
  Again, it does not matter what the folder name is, as long as it does not end with a '/' or anything else after the folder name.

  Data Notes:
  The merged monthly netcdf file used have the following columns:
  - latitude
  - longitude
  - predicted_precip
  - actual_precip (from CHIRPS)
  - date
  - lead_time

  This merged data has been pre-processed with the monthly_merged_data_generation.py script.

  Regions and Seasons Notes:
  The regions_seasons_dict is a dictionary of regions, their seasons, and specific values of month minus lead time.
  Please refer to the above matrix of month minus lead time values to understand how these values were determined,
  as well as how the dictionary works.
  """

  # extract relevant information from the file path
  split = file_path.split('/') # split into list

  file_name = split[-1] # get the file name

  name_split = file_name.split('_') # get the name of the region, i.e [eastern, east, africa]

  region_name = '_'.join(name_split[0:-2]) # combine the name of the region, i.e eastern_east_africa

  new_file_name= file_name.replace('.nc', '_seasonal.nc') # make new file name for saving

  # check if region name is in regions_seasons_dict
  if region_name not in regions_seasons_dict:
      print(f"ValueError: Region '{region_name}' not found in regions_seasons_dict.")
      return

  # open file
  merged_monthly_file = xr.open_dataset(file_path)

  # convert to a dataframe for pre-processing
  merged_monthly_file_df = merged_monthly_file.to_dataframe().reset_index().dropna()

  # seperate month and year into seperate columns
  merged_monthly_file_df['month'] = merged_monthly_file_df['time'].dt.month
  merged_monthly_file_df['year'] = merged_monthly_file_df['time'].dt.year

  # create month_minus_lead_time
  merged_monthly_file_df['month_minus_lead_time'] = merged_monthly_file_df['month'] - merged_monthly_file_df['lead_time']

  # access the dictionary items of the given region
  region_name = 'eastern_east_africa'
  season_data = regions_seasons_dict[region_name]

  # keep track of the number of seasons in the region
  i = 0

  # Iterate through seasons in the region
  for season_name, season_dict in season_data.items():
      months = season_dict['months']

      # Filter only the rows for the relevant season months, i.e only OND months
      seasonal_df = merged_monthly_file_df[merged_monthly_file_df['month'].isin(months)].copy()

      # Create a new column for the current season
      i += 1
      column_name = f"season_label_{i}"
      merged_monthly_file_df[column_name] = None  # initialize empty season column

      # Iterate over each lead category (i.e OND_short)
      for lead_label, lead_values in season_dict.items():
          if lead_label != 'months':
              # Find matching rows based on month_minus_lead_time
              matching_idx = seasonal_df[seasonal_df['month_minus_lead_time'].isin(lead_values)].index

              # Assign the lead_label to the season column in original dataframe
              merged_monthly_file_df.loc[matching_idx, column_name] = lead_label

  # drop uneccesary columns
  merged_monthly_file_df.drop(['month', 'year', 'month_minus_lead_time'], axis=1, inplace=True)

  return merged_monthly_file_df

  # take this dataframe, convert to netcdf, and store in path
  # seasonal_data_xarray = merged_monthly_file_df.set_index(['time', 'latitude', 'longitude']).to_xarray()

  # save to NetCDF
  # seasonal_data_xarray.to_netcdf(f"{save_path}/{new_file_name}")

In [6]:
file_path = '/content/drive/MyDrive/capstone_data/netCDF/eastern_east_africa_CanESM5_merged.nc'
save_path = '/content/drive/MyDrive/capstone_data/netCDF/Seasonal'
eea_seasonal_data = convert_monthly_to_seasonal(file_path, regions_seasons_dict, save_path)

In [7]:
eea_seasonal_data[eea_seasonal_data['season_label_2'] == 'MAM_short']

,lead_time,time,M,latitude,longitude,predicted_precip,precip,season_label_1,season_label_2
24000,0.5,1991-03-01,1.0,-3.5,38.0,0.279488,58.335983,None,MAM_short
24001,0.5,1991-03-01,1.0,-3.5,38.5,0.279488,80.430801,None,MAM_short
24002,0.5,1991-03-01,1.0,-3.5,39.0,0.141809,89.587509,None,MAM_short
24003,0.5,1991-03-01,1.0,-3.5,39.5,0.141809,52.529411,None,MAM_short
24025,0.5,1991-03-01,1.0,-3.0,38.0,0.497000,60.535484,None,MAM_short
...,...,...,...,...,...,...,...,...,...
17807994,3.5,2020-05-01,20.0,8.0,47.5,1.946898,61.963337,None,MAM_short
17807995,3.5,2020-05-01,20.0,8.0,48.0,1.831856,39.457363,None,MAM_short
17807996,3.5,2020-05-01,20.0,8.0,48.5,1.831856,59.003258,None,MAM_short
17807997,3.5,2020-05-01,20.0,8.0,49.0,1.587949,49.644302,None,MAM_short
